# NenuFAR dynamic spectrum (Stokes I and V/I)

This notebook plots high-resolution NenuFAR (New Extension in Nançay Upgrading
LOFAR) dynamic spectra from pre-exported pickle files. NenuFAR covers
**10 - 85 MHz** with native resolutions of 6.10 kHz in frequency and
20.97 ms in time.

The expected pickle file names follow the convention
`<YYYYMMDD>_<group>_stokesI.pkl` and `<YYYYMMDD>_<group>_stokesV_over_I.pkl`,
where `<group>` labels the radio burst (e.g. `typeII`, `typeIII_G1`).

**Workflow:** locate the pickles for the requested date and group &rarr; convert
Stokes I from arbitrary units to dB &rarr; downsample to 1 s &rarr; subtract a
per-channel median background &rarr; plot Stokes I and V/I side-by-side.


In [ ]:
import warnings
warnings.filterwarnings('ignore')

import os
import glob
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import matplotlib.ticker as ticker
from matplotlib.ticker import AutoMinorLocator
from matplotlib.colors import LogNorm
import matplotlib as mpl

# Use the precise matplotlib epoch (avoids ~10 us offsets in old matplotlib).
mpl.rcParams['date.epoch'] = '1970-01-01T00:00:00'
try:
    mdates.set_epoch('1970-01-01T00:00:00')
except RuntimeError:
    pass

# Unified plotting style for all dynamic spectra notebooks.
plt.rcParams['figure.dpi'] = 100
plt.rcParams['savefig.dpi'] = 300
plt.rcParams['figure.facecolor'] = 'white'
plt.rcParams['savefig.facecolor'] = 'white'
plt.rcParams['axes.labelsize'] = 12
plt.rcParams['xtick.labelsize'] = 11
plt.rcParams['ytick.labelsize'] = 11


## Configuration


In [ ]:
# Observation date (no separator) and the burst group label.
mydate = '20250326'

# Possible groups: typeII, typeIII_G1, typeIII_G2, typeIII_G3, typeIII_G4, ...
SRB_groupname = 'typeII'

data_dir = './sample_data/nenufar'
outputs  = './outputs'
os.makedirs(outputs, exist_ok=True)


## Helper functions


In [ ]:
def subtract_background_median(df):
    """
    Subtract a per-channel median background from a dynamic spectrum.

    The function computes the median along the time axis (axis=0) for each
    frequency channel, then subtracts it from every time sample of that
    channel. This is the standard approach for highlighting transient
    emission against a slowly-varying instrumental/sky background.

    Parameters
    ----------
    df : pandas.DataFrame
        DataFrame with shape (n_times, n_freqs). Index is time, columns
        are frequencies.

    Returns
    -------
    pandas.DataFrame
        Same shape as input with the per-channel median removed.
    """
    bkg = np.nanmedian(df.values, axis=0)
    return df - np.tile(bkg, (df.shape[0], 1))


## Locate and load the pickle files


In [ ]:
# Discover all NenuFAR pickles for this date/group.
nenufar_files = sorted(glob.glob(f'{data_dir}/*'))
matching = [f for f in nenufar_files
            if mydate in f and f.endswith(f'{SRB_groupname}.pkl')]

for f in matching:
    print(f)

stokesI_filename  = next(f for f in matching if 'stokesI' in f)
stokesVI_filename = next(f for f in matching if 'stokesV_over_I' in f)

df_int = pd.read_pickle(stokesI_filename)   # Stokes I, arbitrary units
df_pol = pd.read_pickle(stokesVI_filename)  # Stokes V/I, dimensionless

# Convert Stokes I to dB (NenuFAR Stokes I is in arbitrary linear units).
df_int_db = 10 * np.log10(df_int)


In [ ]:
# Report the native cadence for both files (sanity check).
for label, df in [('Stokes I', df_int_db), ('Stokes V/I', df_pol)]:
    df_res = np.median(np.diff(df.columns) * 1e3)
    tm_res = np.median(np.diff(df.index) / np.timedelta64(1, 'ms'))
    print(f'{label}: df = {df_res:.2f} kHz, dt = {tm_res:.2f} ms, shape = {df.shape}')


## Downsample and remove background (Stokes I)


In [ ]:
# Downsample Stokes I to 1 s for plotting. V/I is left at native cadence
# because it is already dimensionless and well-bounded ([-1, 1]).
df_int_1s = df_int_db.resample('1S').mean()

df_int_nobkg = subtract_background_median(df_int_1s)


## Plot Stokes I and V/I side-by-side


In [ ]:
fig = plt.figure(figsize=(14, 6))

# --- Stokes I (background-subtracted) ---
ax1 = fig.add_subplot(121)
vmax = np.nanpercentile(df_int_nobkg.values, 99)
pc1 = ax1.pcolormesh(
    df_int_nobkg.index, df_int_nobkg.columns, df_int_nobkg.T,
    vmin=0, vmax=vmax, cmap='Spectral_r',
)
fig.colorbar(pc1, ax=ax1, pad=0.02, label='dB (background subtracted)')
ax1.set_xlabel(f'Time (UT) on {df_int_nobkg.index[0].date()}')
ax1.set_ylabel('Frequency (MHz)')
ax1.set_ylim(ax1.get_ylim()[::-1])
ax1.yaxis.set_minor_locator(AutoMinorLocator(n=10))
ax1.xaxis_date()
ax1.xaxis.set_major_formatter(mdates.DateFormatter('%H:%M'))
ax1.set_title(f'NenuFAR Stokes I  -  {SRB_groupname}')

# --- Stokes V/I (circular polarisation fraction) ---
ax2 = fig.add_subplot(122)
pc2 = ax2.pcolormesh(
    df_pol.index, df_pol.columns, df_pol.T,
    vmin=-1, vmax=1, cmap='seismic',
)
fig.colorbar(pc2, ax=ax2, pad=0.02, label='V / I')
ax2.set_xlabel(f'Time (UT) on {df_pol.index[0].date()}')
ax2.set_ylabel('Frequency (MHz)')
ax2.set_ylim(ax2.get_ylim()[::-1])
ax2.yaxis.set_minor_locator(AutoMinorLocator(n=10))
ax2.xaxis_date()
ax2.xaxis.set_major_formatter(mdates.DateFormatter('%H:%M'))
ax2.set_title(f'NenuFAR Stokes V/I  -  {SRB_groupname}')

fig.tight_layout()
fig.savefig(f'{outputs}/nenufar_dyspec_{mydate}_{SRB_groupname}.png',
            bbox_inches='tight')
plt.show()
